# Error Analysis — English → Hindi Translation

This notebook inspects the per-sentence outputs from `evaluate.py`
(`data/eval_outputs.tsv`) to understand *where* and *why* the LLM
translator fails, beyond what a single BLEU/chrF++ number tells you.

Run `evaluate.py` first to generate `data/eval_outputs.tsv`:

```bash
python evaluate.py --num_shots 5 --limit 50
```


In [ ]:
import pandas as pd
import sacrebleu

df = pd.read_csv("../data/eval_outputs.tsv", sep="\t")
df.head()

## 1. Per-sentence chrF scores

Compute a chrF score for each individual sentence pair so we can sort
and find the worst-performing translations.

In [ ]:
def sentence_chrf(hyp, ref):
    if not isinstance(hyp, str) or not isinstance(ref, str):
        return 0.0
    return sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score

df["chrf"] = df.apply(lambda row: sentence_chrf(row["hypothesis"], row["reference"]), axis=1)
df = df.sort_values("chrf")
df[["source", "reference", "hypothesis", "chrf"]].head(10)

## 2. Worst-scoring sentences

Manually read through the lowest-scoring rows above and categorize
*why* each one failed. Common categories to look for:

- **Transliteration errors** — English names/brands rendered oddly in Devanagari
- **Idiom mishandling** — literal word-for-word translation of an idiom
- **Register mismatch** — formal English translated too casually or vice versa
- **Omission/hallucination** — words dropped, or extra words added that weren't in the source
- **Long-sentence degradation** — translation quality drops on longer/complex sentences
- **Number/entity errors** — numbers, dates, or named entities mistranslated

Fill in the table below as you inspect rows.

In [ ]:
# Fill this in manually as you inspect failures above.
# Each row: (row_index, failure_category, notes)
error_log = [
    # (3, "idiom mishandling", "translated literally instead of using the natural Hindi idiom"),
]

error_df = pd.DataFrame(error_log, columns=["row_index", "category", "notes"])
error_df

## 3. Failure category breakdown

Once `error_log` above is filled in from manual inspection,
this cell summarizes which failure types are most common —
this is the chart/table to put in your portfolio writeup.

In [ ]:
if len(error_df):
    display(error_df["category"].value_counts())
    error_df["category"].value_counts().plot(kind="barh", title="Failure categories")
else:
    print("Fill in error_log above after inspecting the worst-scoring sentences.")

## 4. Zero-shot vs few-shot comparison (optional)

Run `evaluate.py` twice — once with `--num_shots 0` and once with
`--num_shots 5` — saving each to a different output file, then compare
scores here to show whether few-shot prompting actually helps for
this language pair.

In [ ]:
# Example — adjust filenames to match what you saved from evaluate.py
# zero_shot = pd.read_csv("../data/eval_outputs_zero_shot.tsv", sep="\t")
# few_shot = pd.read_csv("../data/eval_outputs_few_shot.tsv", sep="\t")
#
# for name, d in [("zero-shot", zero_shot), ("few-shot", few_shot)]:
#     bleu = sacrebleu.corpus_bleu(d["hypothesis"].tolist(), [d["reference"].tolist()])
#     print(name, bleu.score)

## 5. Summary

Write a short paragraph here (2-4 sentences) summarizing your findings:
which failure category was most common, whether few-shot prompting helped,
and one concrete idea for improving the prompt or pipeline based on what
you found.